# bootstrap CIs — pilot-2x2

reads results/<exp>/metrics.json (one row per run), writes cis.json for the report.

In [ ]:
# metrics.json come in the form of [run1 config data, run2 config data, etc...]
import json
import numpy as np
import pandas as pd

#
EXPERIMENT = "pilot-2x2"
METRICS_PATH = f"../results/{EXPERIMENT}/metrics.json"
CIS_PATH = f"../results/{EXPERIMENT}/cis.json"

with open(METRICS_PATH) as f:
    rows = pd.DataFrame(json.load(f))

rows

,label,runId,finalGini,offered,accepted,executed,declineRate,unaffordableRate,cost,meanLatencyMs
0,haiku,0b317adc-a6b7-4f43-8d15-fe4d9f975817,0.038501,2,1,1,0.500000,0.000000,0.015104,2340.083333
1,haiku,3f8fe096-ef4d-4a56-8c22-033927277f1f,0.034034,1,1,1,0.000000,0.000000,0.014169,2950.666667
2,4o-mini,ed76d6c6-520e-4301-94d4-3c858ef19f97,0.223228,9,9,8,0.000000,0.111111,0.002170,1828.958333
3,4o-mini,aba80cad-e8f5-4ffc-a7fc-ed2478ba22f1,0.141414,7,5,5,0.285714,0.000000,0.001987,1526.833333


In [2]:
# percentile bootstrap, not a t-interval: no normality assumption, matters at n=2.
# resample size must equal n (that's what makes it a bootstrap, not extra data).
def bootstrap_ci(values, B=10_000, seed=0):
    values = np.asarray(values, dtype=float)
    n = len(values)
    rng = np.random.default_rng(seed)
    means = np.empty(B)
    for i in range(B):
        means[i] = rng.choice(values, size=n, replace=True).mean()
    lo, hi = np.percentile(means, [2.5, 97.5])
    return values.mean(), lo, hi

In [ ]:
# sanity check: zero-variance input -> zero-width CI at the exact mean
known = np.array([10.0, 10.0, 10.0, 10.0, 10.0])
mean, lo, hi = bootstrap_ci(known)
assert abs(mean - 10) < 1e-9
assert abs(hi - lo) < 1e-9
print("sanity check passed")

sanity check passed


In [4]:
METRICS = ["finalGini", "offered", "accepted", "executed",
           "declineRate", "unaffordableRate", "cost", "meanLatencyMs"]

results = []
for label, group in rows.groupby("label"):
    entry = {"label": label, "n": len(group)}
    for m in METRICS:
        mean, lo, hi = bootstrap_ci(group[m].values)
        entry[f"{m}_mean"] = mean
        entry[f"{m}_lo"] = lo
        entry[f"{m}_hi"] = hi
    results.append(entry)

cis = pd.DataFrame(results)
cis

,label,n,finalGini_mean,finalGini_lo,finalGini_hi,offered_mean,offered_lo,offered_hi,accepted_mean,accepted_lo,...,declineRate_hi,unaffordableRate_mean,unaffordableRate_lo,unaffordableRate_hi,cost_mean,cost_lo,cost_hi,meanLatencyMs_mean,meanLatencyMs_lo,meanLatencyMs_hi
0,4o-mini,2,0.182321,0.141414,0.223228,8.0,7.0,9.0,7.0,5.0,...,0.285714,0.055556,0.0,0.111111,0.002078,0.001987,0.002170,1677.895833,1526.833333,1828.958333
1,haiku,2,0.036267,0.034034,0.038501,1.5,1.0,2.0,1.0,1.0,...,0.500000,0.000000,0.0,0.000000,0.014636,0.014169,0.015104,2645.375000,2340.083333,2950.666667


In [5]:
cis.to_json(CIS_PATH, orient="records", indent=2)

n=2 replicates — these CIs describe run-to-run noise in this pilot, not significance (no p-value here). overlapping CIs = can't say the means differ; separated CIs (finalGini below) are suggestive, not proof. more replicates in the full run narrows this.